# EasyRAG Hydration And Citations

## What you will do

- inspect raw ranked records before hydration
- convert them into full text chunks and citation payloads
- verify how backend labels survive into retrieval metadata

## Why this matters

Ranked IDs are not yet usable evidence. Hydration is the step that turns abstract search hits into grounded, inspectable citations.


## Source code anchors

- `easyrag.rag.retrieval.hydration.hydrate_records`
- `easyrag.rag.retrieval.hydration.chunks_to_documents`
- `easyrag.rag.retrieval.hydration.build_citations`
- `easyrag.rag.retrieval.hydration.detect_vector_backend`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.retrieval.hydration import build_citations, chunks_to_documents, detect_vector_backend, hydrate_records


## Deterministic path

We first ask the vector store for raw chunk hits directly. Then we hydrate those hits into the same citation objects that `QueryResult` later exposes.


In [ ]:
hydration_tmp = tempfile.TemporaryDirectory()
hydration_rag = EasyRAG(
    working_dir=hydration_tmp.name,
    workspace="hydration-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(hydration_rag.initialize_storages())
run_sync(
    hydration_rag.ainsert(
        [
            "# Retrieval\nEasyRAG turns ranked records into grounded citations.\n",
            "# Storage\nPaths and titles survive hydration so readers can inspect the source.\n",
        ],
        ids=["doc::retrieval", "doc::storage"],
        file_paths=["docs/retrieval.md", "docs/storage.md"],
    )
)
raw_hits = run_sync(hydration_rag.vector_storage.similarity_search("chunk", "grounded citations", 3))
_print_json(raw_hits)


## Output inspection

Hydration is where ranked records gain full text and citation-ready metadata. The next cell walks that conversion explicitly.


In [ ]:
hydrated_hits = run_sync(hydrate_records(hydration_rag, raw_hits))
hydrated_chunks = run_sync(chunks_to_documents(hydrated_hits))
citations = build_citations(hydrated_chunks)
_print_json({
    "vector_backend": detect_vector_backend(hydrated_hits),
    "hydrated_hits": hydrated_hits,
    "citations": citations,
})


In [ ]:
full_result = run_sync(
    hydration_rag.aquery(
        "How does EasyRAG create grounded citations?",
        QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3),
    )
)
_print_json({"metadata": full_result.metadata, "citations": full_result.citations})


## Provider-backed path

The optional path uses the same hydration helpers with provider-backed retrieval when configured.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-hydration-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(provider_rag.ainsert("# Retrieval\nGrounded citations come from hydrated records.", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
        provider_hits = run_sync(provider_rag.vector_storage.similarity_search("chunk", "grounded citations", 2))
        provider_hydrated = run_sync(hydrate_records(provider_rag, provider_hits))
        _print_json({"backend": detect_vector_backend(provider_hydrated), "hydrated": provider_hydrated})
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Before hydration, you only have scored records. After hydration, you have the text, title, path, and source type needed for citations and rerank. That is why hydration deserves its own inspection step instead of being treated as a trivial formatting detail.


In [ ]:
run_sync(hydration_rag.finalize_storages())
hydration_tmp.cleanup()
print("Cleaned up the hydration workspace.")


## Next steps

- Continue with [04_07_retrieval_failure_cases_and_debugging.ipynb](04_07_retrieval_failure_cases_and_debugging.ipynb) to use these citation artifacts for debugging.
- Read [04-retrieval-overview.md](../../docs/04-retrieval-overview.md) for the retrieval-stage map.
- Revisit [05_01_query_result_to_answer.ipynb](../05_generation/05_01_query_result_to_answer.ipynb) if you want to follow hydrated citations into generation.
